# HeterogeneousMTGP

## 1. このモデルを使う場面

`HeterogeneousMTGP` は、**タスクごとに入力空間が異なる**ときに使うマルチタスクGPです。

例えば、

- タスク0（目的タスク）: 温度・圧力・流量を使う
- タスク1（補助タスク）: 温度・流量だけを使う

のように、タスク間で利用可能な特徴量が一致しない場合を扱えます。

このNotebookでは、タスク0を予測対象とし、タスク1のデータを転移学習的に利用します。

## このNotebookで学ぶこと

- 通常のMultiTaskGPと**異種入力（heterogeneous）MultiTaskGP**の違い
- `feature_indices` でタスク間の共通特徴を対応付ける方法
- 目的タスクの予測平均・不確実性の読み方
- 目的タスクの候補提案と観測追加・再学習
- Mixed版や階層型MultiTaskGPとの違い

このモデルの「heterogeneous」は**タスク間で入力に使う特徴量が異なる**ことを意味します。回帰と分類を同時に扱う意味ではありません。

In [ ]:
import matplotlib.pyplot as plt
import torch

from robotorchan.models import HeterogeneousMTGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. 合成データ

完全な特徴空間を `[x0, x1, x2]` とします。

- タスク0: `[x0, x1, x2]`
- タスク1: `[x0, x2]`

`feature_indices` は各タスクの列が、完全な特徴空間のどこに対応するかを表します。

### 特徴空間の対応表

| 完全特徴空間の列 | 意味 | task 0（目的） | task 1（補助） |
|---|---|---|---|
| 0 | `x0` | 入力列0 | 入力列0 |
| 1 | `x1` | 入力列1 | 観測なし |
| 2 | `x2` | 入力列2 | 入力列1 |

したがって `feature_indices=[[0, 1, 2], [0, 2]]` です。**task 1の入力列1は完全特徴空間の `x2`** に対応します。列番号を誤ると、異なる物理量を同じ特徴として扱うことになるため注意してください。

In [ ]:
def task0_function(X):
    return (
        torch.sin(2 * torch.pi * X[:, 0])
        + 0.7 * (X[:, 1] - 0.5)
        + 0.5 * torch.cos(2 * torch.pi * X[:, 2])
    ).unsqueeze(-1)

def task1_function(X):
    # タスク0と共通する x0, x2 の構造を持つ補助タスク
    return (
        0.85 * torch.sin(2 * torch.pi * X[:, 0])
        + 0.45 * torch.cos(2 * torch.pi * X[:, 1])
        + 0.15
    ).unsqueeze(-1)

train_X0 = torch.rand(18, 3)
train_Y0 = task0_function(train_X0) + 0.04 * torch.randn(18, 1)

train_X1 = torch.rand(28, 2)
train_Y1 = task1_function(train_X1) + 0.04 * torch.randn(28, 1)

feature_indices = [
    [0, 1, 2],  # タスク0: x0, x1, x2
    [0, 2],     # タスク1: x0, x2
]

print(train_X0.shape, train_X1.shape)
print(train_Y0.shape, train_Y1.shape)

## 3. モデル構築

BoTorch / robotorchan の `HeterogeneousMTGP` では **タスク0が目的タスク**です。
`train_Xs` 自体には task feature を含めません。

### データshapeを確認する

目的タスクには3変数の18観測、補助タスクには2変数の28観測を用意します。`train_Xs` と `train_Ys` は**タスクごとのテンソルをリストで渡す**形式です。

`full_feature_dim=3` は完全特徴空間の次元数です。`train_Xs` にtask ID列を追加しません。モデル内部でタスクの対応付けを行います。

In [ ]:
model = HeterogeneousMTGP(
    train_Xs=[train_X0, train_X1],
    train_Ys=[train_Y0, train_Y1],
    train_Yvars=None,
    feature_indices=feature_indices,
    full_feature_dim=3,
    use_saas_prior=False,
)

## 4. robotorchan のrawデータAPI

通常の `raw_train_X` ではなく、タスクごとのデータを `raw_train_Xs` / `raw_train_Ys` として保持します。

### 情報共有と転移の前提

目的タスクの応答は `x0`, `x1`, `x2` に依存します。一方、補助タスクは共通の `x0`, `x2` に関連する応答を持ちます。モデルは**共通する特徴とタスク間の相関**を通じて補助データを活用することを狙います。

補助データを増やせば必ず目的タスクの予測が良くなるわけではありません。タスク間の関連が弱い場合は、負の転移も起こり得ます。

In [ ]:
print("supports_mll:", model.supports_mll)
print("supports_fit:", model.supports_fit)
print("raw_train_Xs:", [x.shape for x in model.raw_train_Xs])
print("raw_train_Ys:", [y.shape for y in model.raw_train_Ys])
print("raw_train_Yvars:", model.raw_train_Yvars)
print("raw_data keys:", model.raw_data.keys())

## 5. 学習

### `fit()` による学習

`HeterogeneousMTGP` は `fit()` から既存の `make_mll()` と BoTorch の `fit_gpytorch_mll()` を利用して学習します。ここでは `use_saas_prior=False` とし、疎性事前分布の影響を切り分けています。

学習後は `eval()` に切り替えて予測します。これは一般的なExact GPの学習フローと同じです。

In [ ]:
model.fit()
model.eval()

## 6. 目的タスクのposterior

posteriorはタスク0に対して計算します。
現行BoTorch APIでは、入力末尾に task feature `0` を付けます。

### 予測時のtask feature

学習時はタスクごとの `train_Xs` にtask IDを付けませんが、**事後予測の入力にはtask ID列が必要**です。このNotebookでは末尾に0を追加し、目的タスクを指定します。

`test_X` は `(120, 3)`、`test_X_with_task` は `(120, 4)` です。予測は `x0` を動かし、`x1=x2=0.5` に固定した**1次元スライス**です。

In [ ]:
x0 = torch.linspace(0.0, 1.0, 120)
test_X = torch.stack(
    [
        x0,
        torch.full_like(x0, 0.5),
        torch.full_like(x0, 0.5),
    ],
    dim=-1,
)
test_X_with_task = torch.cat(
    [test_X, torch.zeros(test_X.shape[0], 1)],
    dim=-1,
)

with torch.no_grad():
    posterior = model.posterior(test_X_with_task)
    mean = posterior.mean.squeeze(-1)
    std = posterior.variance.clamp_min(0).sqrt().squeeze(-1)

truth = task0_function(test_X).squeeze(-1)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x0, truth, label="真値")
plt.plot(x0, mean, label="HeterogeneousMTGP")
plt.fill_between(
    x0,
    mean - 1.96 * std,
    mean + 1.96 * std,
    alpha=0.2,
    label="95%区間",
)
plt.xlabel("x0（x1=x2=0.5で固定）")
plt.ylabel("y")
plt.legend()
plt.show()

### 予測区間の読み方

予測平均に加えて、`posterior.variance` から標準偏差を計算しています。図の `mean ± 1.96 × std` はモデル上の**潜在関数に対する近似95%事後区間**です。観測ノイズ込みの測定値区間や、厳密な被覆保証ではありません。

合成データでは真の関数を描けますが、実データでは未観測点の真値は通常わかりません。

## 7. 使い分け

### HeterogeneousMTGPが向くケース
- タスクごとに入力次元や利用可能な特徴量が違う
- 補助タスクの知識を目的タスクへ転移したい
- 異なる実験系・設備・シミュレータ間で共通特徴が一部だけ存在する

### 通常のMultiTaskGPが向くケース
- 全タスクが同じ特徴空間を持つ
- task featureを付けるだけで表現できる

### 注意
`feature_indices` は「各タスク内の列が完全特徴空間のどこに対応するか」を正しく指定する必要があります。
また、このモデルのposteriorは目的タスク（task 0）に限定されます。

### 追加実装例：目的タスクの予測誤差を確認する

予測曲線だけでは誤差の大きさを比較しにくいため、この1次元スライス上で真の関数とのRMSEを計算します。これは**このスライスに限定した評価**であり、3次元入力空間全体の汎化性能を示すものではありません。

In [ ]:
slice_rmse = torch.sqrt(torch.mean((mean - truth) ** 2))
print("task 0 slice RMSE:", float(slice_rmse))
print("posterior mean shape:", tuple(mean.shape))
print("posterior std shape:", tuple(std.shape))

## Target-task candidate proposal

タスク0の事後平均を最大化する候補を生成します。補助タスクとは入力次元が異なるため、候補には全特徴量とタスク識別子を明示します。

In [ ]:
with torch.no_grad():
    scores = model.posterior(test_X_with_task).mean.squeeze(-1)
    best_index = scores.argmax()
    candidate = test_X_with_task[best_index:best_index + 1]
candidate_Y = task0_function(candidate[:, :3])
assert candidate.shape == (1, 4)
assert candidate[0, -1].item() == 0.0
print("candidate:", candidate)
print("predicted objective:", scores[best_index])
print("evaluated objective:", candidate_Y)


### 候補提案はどの探索方法か

既存の例は120点の候補から**事後平均が最大の点**を選びます。EIやUCBのような探索と活用のバランスを取る獲得関数ではありません。また `x1=x2=0.5` に固定しているため、3次元全域を最適化しているわけではありません。

候補の末尾のtask IDは0に固定します。候補を実際に評価するときはtask IDを除いた3列を目的関数に渡します。

## 候補の観測追加と再学習

目的タスクで得た観測をタスク0の学習データに追加し、補助タスクのデータは維持して再学習します。ここで扱う「heterogeneous」はタスク間の入力特徴量の違いであり、回帰・分類の混在出力とは異なります。


In [ ]:
observed_X0 = candidate[:, :3].detach()
observed_Y0 = candidate_Y.detach()
updated_X0 = torch.cat([train_X0, observed_X0], dim=0)
updated_Y0 = torch.cat([train_Y0, observed_Y0], dim=0)

updated_model = HeterogeneousMTGP(
    train_Xs=[updated_X0, train_X1],
    train_Ys=[updated_Y0, train_Y1],
    train_Yvars=None,
    feature_indices=feature_indices,
    full_feature_dim=3,
    use_saas_prior=False,
)
updated_model.fit()
updated_model.eval()

assert updated_X0.shape[0] == train_X0.shape[0] + 1
assert updated_model.posterior(candidate).mean.shape == torch.Size([1, 1])
print("updated objective-task observations:", updated_X0.shape[0])


### 観測追加と再学習の流れ

目的タスクの候補を評価したら、**目的タスクの `train_X0` と `train_Y0` だけを更新**します。補助タスクの観測はそのまま保持します。

再学習時には `feature_indices` と `full_feature_dim` を変えずに新しいモデルを構築します。これにより、異種入力の対応関係を維持したまま目的タスクの情報を追加できます。

## MixedHeterogeneousMTGP：カテゴリ付き異種入力

目的タスクには連続2列とカテゴリ1列、補助タスクには連続1列と同じカテゴリ列を与えます。カテゴリは有限値として列挙し、目的タスクのPosterior平均から候補を選択します。


In [ ]:
from robotorchan.models import MixedHeterogeneousMTGP

mixed_X0 = torch.cat(
    [train_X0[:, :2], (train_X0[:, 2:3] > 0.5).to(train_X0)],
    dim=-1,
)
mixed_X1 = torch.cat(
    [train_X1[:, :1], (train_X1[:, 1:2] > 0.5).to(train_X1)],
    dim=-1,
)

def mixed_target(X):
    return (
        torch.sin(2 * torch.pi * X[:, :1])
        + 0.5 * X[:, 1:2]
        + 0.2 * X[:, 2:3]
    )

mixed_Y0 = mixed_target(mixed_X0)
mixed_Y1 = (
    0.8 * torch.sin(2 * torch.pi * mixed_X1[:, :1])
    + 0.2 * mixed_X1[:, 1:2]
)
mixed_model = MixedHeterogeneousMTGP(
    train_Xs=[mixed_X0, mixed_X1],
    train_Ys=[mixed_Y0, mixed_Y1],
    train_Yvars=None,
    feature_indices=[[0, 1, 2], [0, 2]],
    full_feature_dim=3,
    cat_dims=[2],
    use_saas_prior=False,
)
mixed_model.fit()
mixed_model.eval()

mixed_grid = torch.linspace(0.0, 1.0, 41)
mixed_candidates = torch.cat(
    [
        torch.stack(
            [
                mixed_grid,
                torch.full_like(mixed_grid, 0.5),
                torch.full_like(mixed_grid, float(category)),
                torch.zeros_like(mixed_grid),
            ],
            dim=-1,
        )
        for category in (0, 1)
    ],
    dim=0,
)
with torch.no_grad():
    mixed_scores = mixed_model.posterior(
        mixed_candidates
    ).mean.squeeze(-1)
mixed_index = int(mixed_scores.argmax().item())
mixed_candidate = mixed_candidates[mixed_index:mixed_index + 1]
mixed_observed = mixed_target(mixed_candidate[:, :3])
assert mixed_candidate.shape == (1, 4)
assert mixed_candidate[0, 2].item() in (0.0, 1.0)
assert mixed_candidate[0, 3].item() == 0.0
print("Mixed heterogeneous candidate:", mixed_candidate)
print("Evaluated objective:", mixed_observed)


### Mixed HeterogeneousMTGP

Mixed版では、タスク間で利用できる特徴量が異なることに加え、カテゴリ変数も入力に含められます。`cat_dims=[2]` は**完全特徴空間のカテゴリ列**を指します。

候補生成ではカテゴリ0と1を列挙し、連続変数を含む候補集合から予測平均が最大の点を選びます。カテゴリは連続値として補間しません。

## 階層型MultiTaskGPの候補提案

方式0では温度のみ、方式1では圧力のみを有効にします。task 1の事後平均を比較し、Mixed型では独立したカテゴリ設計変数も列挙します。


In [ ]:
from robotorchan.models import (
    HierarchicalConditionalKernelMultiTaskGP,
    MixedHierarchicalConditionalKernelMultiTaskGP,
)

hierarchy = {1: {0: [2], 1: [3]}}
base_grid = torch.linspace(0.05, 0.95, 10)
hierarchical_design = torch.cat(
    [
        torch.stack(
            [
                base_grid,
                torch.full_like(base_grid, float(branch)),
                base_grid if branch == 0 else torch.full_like(base_grid, 0.5),
                base_grid if branch == 1 else torch.full_like(base_grid, 0.5),
            ],
            dim=-1,
        )
        for branch in (0, 1)
    ],
    dim=0,
)

def hierarchical_target(X):
    active = torch.where(X[:, 1:2] == 0, X[:, 2:3], X[:, 3:4])
    return torch.sin(2 * torch.pi * X[:, :1]) + 0.3 * active


hierarchical_training = torch.cat(
    [
        torch.cat(
            [hierarchical_design, torch.zeros(len(hierarchical_design), 1)],
            dim=-1,
        ),
        torch.cat(
            [hierarchical_design, torch.ones(len(hierarchical_design), 1)],
            dim=-1,
        ),
    ],
    dim=0,
)
hierarchical_Y = torch.cat(
    [
        hierarchical_target(hierarchical_design),
        0.8 * hierarchical_target(hierarchical_design) + 0.2,
    ],
    dim=0,
)
hierarchical_models = [
    (
        "Hierarchical multi-task",
        HierarchicalConditionalKernelMultiTaskGP(
            train_X=hierarchical_training,
            train_Y=hierarchical_Y,
            task_feature=4,
            hierarchical_dependencies=hierarchy,
            output_tasks=[1],
            use_saas_prior=False,
        ),
    ),
]

mixed_hierarchy_design = torch.cat(
    [
        hierarchical_design,
        (hierarchical_design[:, :1] > 0.5).to(hierarchical_design),
    ],
    dim=-1,
)
mixed_hierarchy_training = torch.cat(
    [
        torch.cat(
            [mixed_hierarchy_design, torch.zeros(len(mixed_hierarchy_design), 1)],
            dim=-1,
        ),
        torch.cat(
            [mixed_hierarchy_design, torch.ones(len(mixed_hierarchy_design), 1)],
            dim=-1,
        ),
    ],
    dim=0,
)
mixed_hierarchy_Y = hierarchical_Y + 0.1 * mixed_hierarchy_training[:, 4:5]
hierarchical_models.append(
    (
        "Mixed hierarchical multi-task",
        MixedHierarchicalConditionalKernelMultiTaskGP(
            train_X=mixed_hierarchy_training,
            train_Y=mixed_hierarchy_Y,
            task_feature=5,
            hierarchical_dependencies=hierarchy,
            cat_dims=[4],
            output_tasks=[1],
            use_saas_prior=False,
        ),
    )
)

for name, surrogate in hierarchical_models:
    surrogate.fit()
    surrogate.eval()
    pool = hierarchical_design
    if name.startswith("Mixed"):
        pool = torch.cat(
            [
                torch.cat(
                    [
                        hierarchical_design,
                        torch.full(
                            (len(hierarchical_design), 1), float(category)
                        ),
                    ],
                    dim=-1,
                )
                for category in (0, 1)
            ],
            dim=0,
        )
    with torch.no_grad():
        scores = surrogate.posterior(pool).mean.squeeze(-1)
    best = int(scores.argmax().item())
    candidate = pool[best:best + 1]
    observed = 0.8 * hierarchical_target(candidate[:, :4]) + 0.2
    if name.startswith("Mixed"):
        observed = observed + 0.1 * candidate[:, 4:5]
        assert candidate[0, 4].item() in (0.0, 1.0)
    assert candidate[0, 1].item() in (0.0, 1.0)
    assert torch.isfinite(observed).all()
    print(name, "candidate:", candidate, "objective:", observed)


### 階層型MultiTaskGPとの違い

**HeterogeneousMTGP**はタスクごとに**観測・利用可能な入力特徴量が異なる**問題を扱います。一方、**HierarchicalConditionalKernelMultiTaskGP**は、親変数の選択によって**有効になる子変数が変わる**探索空間を扱います。

どちらもMulti-taskの情報共有を使いますが、入力の違いが生じる理由が異なります。後者の例は別のモデルファミリーを比較する参考例です。